# 用 Iroh 学 Rust：从语法到 P2P

这是一份可执行教程，同时回答两个问题：

1. Rust 的语法为什么这样设计？
2. Iroh 如何把这些语言能力组合成加密、直连优先、relay 兜底的 P2P 应用？

版本基线（2026-07-23 核对）：`iroh 1.0.3`、`iroh-gossip 0.101.0`、`iroh-tickets 1.0.0`，Rust 1.91+。

> 本教程不把 Iroh 的 Endpoint 身份等同于 Pharos 的授权。传输认证、网络可达性、应用成员资格是三层不同概念。

## 0. Notebook 使用方式

Evcxr 会保留前一个 cell 的变量。若你重复定义同名类型或运行中途留下任务，使用 **Kernel → Restart Kernel**，然后从头执行。

下面的 `:dep` 是 Evcxr 指令，不是普通 Rust 语法。第一次执行会下载并编译依赖。

In [ ]:
:dep anyhow = "1.0"
:dep iroh = "=1.0.3"
:dep iroh-gossip = "=0.101.0"
:dep iroh-tickets = "=1.0.0"
:dep tokio = { version = "1.49", features = ["macros", "rt-multi-thread", "time"] }

## 1. Rust 首先是一门“表达式语言”

- `let` 创建绑定；默认不可变，`mut` 才允许修改。
- 代码块的最后一行不写分号时，它就是这个代码块的返回值。
- `if`、`match` 和 `{ ... }` 都能产生值。
- `&str` 是借用的字符串切片，`String` 是拥有内存的可增长字符串。

这种设计让状态变化显式，也让编译器能在网络代码并发执行前发现大量数据竞争。

In [ ]:
let endpoint_name: &str = "phone-16";
let mut connected_peers: u32 = 1;
connected_peers += 1;

let health = if connected_peers > 0 { "online" } else { "offline" };
let summary = format!("{endpoint_name}: {health}, peers={connected_peers}");
summary

### 所有权与借用

`String` 离开作用域时自动释放。把它赋给另一个变量通常会 **move** 所有权；传 `&value` 则只是临时借用。

网络系统中，这能阻止两个任务无约束地同时修改同一缓冲区。需要共享所有权时常用 `Arc<T>`；需要共享可变状态时再组合锁或 channel。不要一上来就用 `Arc<Mutex<_>>`，消息传递通常更清楚。

In [ ]:
fn byte_count(message: &str) -> usize {
    message.len()
}

let owned = String::from("hello mesh");
let count = byte_count(&owned); // 借用，owned 仍然可用
(owned, count)

## 2. enum、match、Result 与 `?`

`enum` 表示“有限个可能状态”，每个分支还能携带不同数据。Rust 的 `Result<T, E>` 本身就是：

```rust
enum Result<T, E> { Ok(T), Err(E) }
```

`?` 在成功时取出 `T`，失败时把错误提前返回。它不是忽略错误，而是简洁地传播错误。生产网络代码应给错误补上下文，并对连接、读取和消息大小设置边界。

In [ ]:
#[derive(Debug)]
enum PathKind {
    Direct { latency_ms: u32 },
    Relay(String),
    Offline,
}

fn describe_path(path: &PathKind) -> String {
    match path {
        PathKind::Direct { latency_ms } => format!("direct ({latency_ms} ms)"),
        PathKind::Relay(region) => format!("relay via {region}"),
        PathKind::Offline => "offline".to_owned(),
    }
}

describe_path(&PathKind::Direct { latency_ms: 18 })

In [ ]:
fn parse_port(text: &str) -> anyhow::Result<u16> {
    let port: u16 = text.parse()?;
    anyhow::ensure!(port != 0, "port 0 is reserved for automatic binding here");
    Ok(port)
}

parse_port("443")?

### struct、impl、泛型、Option、闭包与迭代器

- `struct` 把有关联的字段组成自己的类型；`impl` 为它添加方法。
- `Option<T>` 是 `Some(T)` 或 `None`，用类型系统表达“可能没有”，避免 null。
- `<T>` 是泛型：一份算法适用于多种具体类型，同时保持编译期检查。
- `|x| ...` 是闭包；`.map()`、`.filter()`、`.collect()` 组成惰性 iterator pipeline。
- 引用带 lifetime，写作 `&'a T`。大部分简单函数由编译器推断；只有多个引用的关系不明确时才需显式标注。

Iroh API 中随处可见这些设计：可选 relay、按类型区分的地址、对 peer 集合的 iterator，以及接受 handler 的泛型 Router。

In [ ]:
#[derive(Debug)]
struct Peer {
    name: String,
    latency_ms: Option<u32>,
}

impl Peer {
    fn is_online(&self) -> bool {
        self.latency_ms.is_some()
    }
}

let peers = vec![
    Peer { name: "mac-mini".into(), latency_ms: Some(12) },
    Peer { name: "iphone".into(), latency_ms: None },
];

let online_names: Vec<&str> = peers.iter()
    .filter(|peer| peer.is_online())
    .map(|peer| peer.name.as_str())
    .collect();
online_names

## 3. async/.await：等待，不等于阻塞线程

`async fn` 返回一个 Future。只有 `.await` 时它才推进；等待网络时，执行器可以运行别的任务。`tokio::spawn` 要求任务拥有捕获的数据，因此常见 `async move`。

始终给外部网络等待加 timeout。Notebook 尤其不应留下无限 accept loop。

In [ ]:
use std::time::Duration;
use tokio::time::{sleep, timeout};

async fn simulated_dial() -> &'static str {
    sleep(Duration::from_millis(50)).await;
    "connected"
}

timeout(Duration::from_secs(1), simulated_dial()).await?

## 4. Iroh 的核心模型

官方文档把 `Endpoint` 定义为应用连接与接收连接的核心对象。一个 Endpoint：

- 用 Ed25519 密钥对的公开部分形成稳定 `EndpointId`；重启后保持身份需要持久化 `SecretKey`。
- 在端到端加密 QUIC 上建立连接。
- 优先寻找直连路径，必要时通过 relay 传输；relay 不获得明文。
- 通常一个应用进程使用一个 Endpoint，再用 ALPN/Router 承载多个协议。

`EndpointId` 是“你是谁”，`EndpointAddr`/discovery/ticket 是“现在怎样找到你”。应用层仍必须判断这个 key 是否属于当前可信 Mesh。

In [ ]:
use iroh::{Endpoint, endpoint::presets};

let identity_demo = Endpoint::bind(presets::N0).await?;
let endpoint_id = identity_demo.id();
let endpoint_addr = identity_demo.addr();
println!("EndpointId: {endpoint_id}");
println!("EndpointAddr: {endpoint_addr:?}");
identity_demo.close().await;

### Rust 语法观察

- `use a::{b, c}` 一次导入多个名字。
- `Endpoint::bind(...)` 的 `::` 调用类型关联函数；`endpoint.id()` 的 `.` 调用实例方法。
- `println!(...)` 末尾的 `!` 表示宏。
- `{:?}` 使用 `Debug` trait；`{endpoint_id}` 使用 `Display` trait。
- `close().await` 展示资源的显式、异步优雅关闭。

## 5. 实验：同一进程中的两个真实 Endpoint

下面不是 mock：server 与 client 会建立真实 Iroh/QUIC 连接。Router 根据 ALPN 把连接交给 `Echo`。

ALPN 建议包含产品、协议和主版本，例如 `/pharos/sync/2`。不兼容版本应该协商失败，而不是悄悄误读数据。

`ProtocolHandler` 是 trait（类似 Swift protocol/interface）；`impl ProtocolHandler for Echo` 为自己的类型实现协议行为。`#[derive(Debug, Clone)]` 让编译器生成常用 trait 实现。

In [ ]:
use iroh::{
    endpoint::Connection,
    protocol::{AcceptError, ProtocolHandler, Router},
};

const ECHO_ALPN: &[u8] = b"/pharos/tutorial/echo/1";
const MAX_BYTES: usize = 64 * 1024;

#[derive(Debug, Clone)]
struct Echo;

impl ProtocolHandler for Echo {
    async fn accept(&self, connection: Connection) -> Result<(), AcceptError> {
        let (mut send, mut receive) = connection.accept_bi().await?;
        let message = receive
            .read_to_end(MAX_BYTES)
            .await
            .map_err(AcceptError::from_err)?;
        send.write_all(&message)
            .await
            .map_err(AcceptError::from_err)?;
        send.finish()?;
        connection.closed().await;
        Ok(())
    }
}

In [ ]:
let server_endpoint = Endpoint::bind(presets::N0).await?;
let server_address = server_endpoint.addr();
let server = Router::builder(server_endpoint)
    .accept(ECHO_ALPN, Echo)
    .spawn();

let client = Endpoint::bind(presets::N0).await?;
let connection = timeout(
    Duration::from_secs(10),
    client.connect(server_address, ECHO_ALPN),
).await??;

let (mut send, mut receive) = connection.open_bi().await?;
send.write_all(b"hello from the notebook").await?;
send.finish()?;

let reply = timeout(
    Duration::from_secs(10),
    receive.read_to_end(MAX_BYTES),
).await??;
println!("reply = {}", String::from_utf8(reply)?);

connection.close(0u8.into(), b"done");
client.close().await;
server.shutdown().await?;

### 为什么 `open_bi` 之后还要写数据？

Iroh 官方 API 文档特别提醒：仅调用 `open_bi()` 不足以让对端的 `accept_bi()` 返回；发送端必须实际写入数据。这个细节很容易制造“双方都在线但卡住”的假死。

同样，`read_to_end(MAX_BYTES)` 的上限是安全边界。不要对不可信 peer 使用无界读取。

`map_err(AcceptError::from_err)` 展示了 Rust 的显式错误转换：stream 错误和 Router handler 错误是不同类型，只有转换后 `?` 才能提前返回正确的错误类型。

## 6. Ticket：bootstrap，不是长期身份数据库

Ticket 是可复制、二维码化或通过 AirDrop/Mail 发送的 bootstrap token。Endpoint ticket 包含 EndpointId 和当时已知的可达地址。

注意：

- 地址会过期；长期记录优先保存稳定 EndpointId，并通过 discovery 获取新地址。
- ticket 可重复使用，除非应用自己实现单次消费、过期和撤销。
- ticket 可能暴露 IP 地址，也可能携带能力或秘密；按 bearer credential 保护。
- “能连上 ticket 里的 endpoint”不等于“已加入 Pharos Mesh”。加入还需要签名的成员资格转换。

In [ ]:
use iroh_tickets::endpoint::EndpointTicket;

let ticket_endpoint = Endpoint::bind(presets::N0).await?;
let ticket = EndpointTicket::new(ticket_endpoint.addr());
let encoded = ticket.to_string();
let decoded: EndpointTicket = encoded.parse()?;

assert_eq!(ticket, decoded);
println!("ticket prefix: {}…", &encoded[..encoded.len().min(48)]);
ticket_endpoint.close().await;

## 7. Gossip：广播覆盖层，不是万能数据库

`iroh-gossip` 让订阅同一 `TopicId` 的 peers 在广播树上传播消息。节点仍需至少知道 bootstrap peers 才能加入。典型流程是：

```rust
let endpoint = Endpoint::bind(presets::N0).await?;
let gossip = Gossip::builder().spawn(endpoint.clone());
let router = Router::builder(endpoint)
    .accept(GOSSIP_ALPN, gossip.clone())
    .spawn();
let (sender, receiver) = gossip
    .subscribe_and_join(topic_id, bootstrap_peer_ids)
    .await?
    .split();
sender.broadcast(message.into()).await?;
router.shutdown().await?;
```

Rust 语法点：泛型集合能让 `bootstrap_peer_ids` 的元素类型在编译期确定；`.split()` 用 tuple destructuring 分开收发端；receiver 通常作为 Stream，用 `while let Some(event) = receiver.next().await` 消费。

适合：presence hint、状态失效通知、短小的实时 fan-out、唤醒同步。  
不适合单独承担：永久聊天历史、成员资格真相、撤销审计、必须送达的命令。peer 离线时需要 durable log + reconciliation。

## 8. 从官方能力到 Pharos/MeshKit

| Iroh 概念 | 在 Pharos/MeshKit 中的职责 | 不能替代什么 |
|---|---|---|
| `SecretKey` / `EndpointId` | 设备传输身份；私钥安全持久化 | 用户显示名、agent session ID、Mesh admin 权限 |
| direct + relay + discovery | 找到 peer 并保持跨网络可达 | 成员授权、冲突解决 |
| `Router` + versioned ALPN | 同一 Endpoint 复用 sync、pairing、diagnostics 等协议 | payload schema migration |
| bidirectional QUIC stream | 有响应的 RPC、增量同步、控制命令 | 离线必达语义 |
| ticket | QR/magic-link 配对 bootstrap | 永久设备主键、单次邀请状态机 |
| gossip topic | presence/invalidation/wakeup 的快速扩散 | SQLite 事件日志和签名成员 epoch |

去中心化不等于没有规则。每台设备可以拥有完整副本，但仍需确定性的事件身份、签名验证、成员 epoch 和冲突策略，才能避免两个离线写入在重连时互相覆盖。

## 9. 生产检查表

- 持久化 `SecretKey`，否则每次启动都会像新设备；在 Apple 平台放 Keychain。
- 长期关系存 EndpointId；ticket/地址只作为可刷新 hint。
- 一个进程优先一个 Endpoint；用 versioned ALPN 组合协议。
- 连接、读写、消息大小、并发 stream 数都有 timeout/limit。
- handler 处理 payload 前，先把 remote EndpointId 映射到当前授权成员 epoch。
- pairing protocol 与 normal sync protocol 使用不同 admission policy。
- gossip 只发轻量提示；事实从可重放、签名、可校验的 durable store 对账。
- shutdown Router 和 Endpoint，让 peer 及时观察离线，测试也不会遗留任务。
- 记录连接尝试、直连/relay 路径、重连耗时和拒绝原因，但避免日志泄露 ticket/私钥/完整 IP。
- 用网络切换、peer 睡眠、地址过期、重复消息、乱序消息和并发成员变更做 fault injection。

## 10. 练习

1. **Rust 入门：** 给 `PathKind` 增加 `Unknown` 分支。观察编译器如何要求 `match` 穷尽所有状态。
2. **错误处理：** 把 echo 的 timeout 改成 1 毫秒，用 `anyhow::Context` 添加“connect”或“read reply”上下文。
3. **协议版本：** 让 client 使用 `/pharos/tutorial/echo/2`，观察 ALPN 不匹配如何失败。
4. **资源边界：** 发送超过 `MAX_BYTES` 的消息，确认 receiver 拒绝无界内存增长。
5. **身份：** 使用 `SecretKey::generate()`，重建两个 Endpoint 时复用同一个 key，验证 EndpointId 稳定。
6. **两进程实验：** 按官方 Quickstart 把 receiver 和 sender 分开运行，通过 ticket 连接两台真实设备。
7. **架构判断：** 为 presence、chat message、membership transition、poke command 分别选择 stream、gossip 或 durable sync，并写出失败恢复方式。

建议顺序：先完成 1–4，再把同源 `src/bin/local-echo.rs` 拆成真正的 sender/receiver 两个 binary。

## 11. 官方资料索引

- [What is Iroh?](https://docs.iroh.computer/what-is-iroh)
- [Quickstart](https://docs.iroh.computer/quickstart)
- [Connect two endpoints](https://docs.iroh.computer/connect-two-endpoints)
- [Endpoint concepts](https://docs.iroh.computer/concepts/endpoints)
- [Ticket concepts](https://docs.iroh.computer/concepts/tickets)
- [Protocols and ALPN](https://docs.iroh.computer/concepts/protocols)
- [Using QUIC](https://docs.iroh.computer/protocols/using-quic)
- [Gossip](https://docs.iroh.computer/connecting/gossip)
- [Security and privacy](https://docs.iroh.computer/deployment/security-privacy)
- [Troubleshooting](https://docs.iroh.computer/troubleshooting)
- [`iroh` 1.0.3 API](https://docs.rs/iroh/1.0.3/iroh/)
- [`iroh-gossip` 0.101.0 API](https://docs.rs/iroh-gossip/0.101.0/iroh_gossip/)

阅读方法：概念页回答“为什么”，docs.rs 回答“准确类型和方法是什么”，官方 examples 回答“如何组合”，最后再由 Pharos 的 threat model 决定授权和持久化语义。